# Comparison: ResNet18 (Skip) vs CNN18 (No Skip)

This notebook compares both segmentation experiments using saved logs and qualitative prediction exports.

Run both training notebooks first so the required outputs exist under outputs/.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path("outputs")
COMP_DIR = BASE_DIR / "comparison_resnet18_vs_cnn18"
COMP_PLOT_DIR = COMP_DIR / "plots"
COMP_TABLE_DIR = COMP_DIR / "tables"
for path in [COMP_PLOT_DIR, COMP_TABLE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

VARIANTS = [
    {"name": "ResNet18 (with skip)", "tag": "resnet18_seg", "color": "#0f766e", "linestyle": "-"},
    {"name": "CNN18 (no skip)", "tag": "cnn18_no_skip_seg", "color": "#b45309", "linestyle": "--"},
]

logs = {}
for variant in VARIANTS:
    log_path = BASE_DIR / variant["tag"] / "logs" / f"{variant['tag']}_metrics.csv"
    if not log_path.exists():
        raise FileNotFoundError(f"Missing log file: {log_path}. Run the corresponding training notebook first.")
    logs[variant["tag"]] = pd.read_csv(log_path)
    print(f"Loaded {variant['name']} -> {log_path}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for variant in VARIANTS:
    df = logs[variant["tag"]]

    axes[0].plot(df["epoch"], df["val_loss"], label=variant["name"], color=variant["color"], linestyle=variant["linestyle"], linewidth=2)
    axes[1].plot(df["epoch"], df["val_dice"], label=variant["name"], color=variant["color"], linestyle=variant["linestyle"], linewidth=2)
    axes[2].plot(df["epoch"], df["val_iou"], label=variant["name"], color=variant["color"], linestyle=variant["linestyle"], linewidth=2)

axes[0].set_title("Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].set_title("Validation Dice")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Dice")
axes[1].grid(alpha=0.3)
axes[1].legend()

axes[2].set_title("Validation IoU")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("IoU")
axes[2].grid(alpha=0.3)
axes[2].legend()

plt.tight_layout()
curve_path = COMP_PLOT_DIR / "comparison_curves.png"
plt.savefig(curve_path, dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved comparison curves: {curve_path}")

In [ ]:
def epoch_to_threshold(series: pd.Series, threshold: float):
    idx = np.where(series.values >= threshold)[0]
    if len(idx) == 0:
        return np.nan
    return int(idx[0] + 1)


rows = []
for variant in VARIANTS:
    df = logs[variant["tag"]]
    best_idx = int(df["val_dice"].idxmax())

    rows.append({
        "model": variant["name"],
        "best_val_dice": float(df["val_dice"].max()),
        "best_val_iou": float(df.loc[best_idx, "val_iou"]),
        "best_epoch": best_idx + 1,
        "final_val_dice": float(df["val_dice"].iloc[-1]),
        "final_val_iou": float(df["val_iou"].iloc[-1]),
        "final_val_loss": float(df["val_loss"].iloc[-1]),
        "avg_epoch_time_s": float(df["time_s"].mean()),
        "epoch_to_dice_0_50": epoch_to_threshold(df["val_dice"], 0.50),
        "epoch_to_dice_0_60": epoch_to_threshold(df["val_dice"], 0.60),
    })

summary_df = pd.DataFrame(rows).sort_values("best_val_dice", ascending=False).reset_index(drop=True)
display(summary_df)

summary_csv = COMP_TABLE_DIR / "summary_metrics.csv"
summary_df.to_csv(summary_csv, index=False)
print(f"Saved summary table: {summary_csv}")

winner = summary_df.iloc[0]
print(f"Best model by val Dice: {winner['model']} (Dice={winner['best_val_dice']:.4f})")

In [ ]:
resnet_npz = BASE_DIR / "resnet18_seg" / "plots" / "resnet18_seg_qualitative.npz"
cnn_npz = BASE_DIR / "cnn18_no_skip_seg" / "plots" / "cnn18_no_skip_seg_qualitative.npz"

if not resnet_npz.exists() or not cnn_npz.exists():
    print("Qualitative npz files are missing. Run the final cell in both training notebooks first.")
else:
    resnet_data = np.load(resnet_npz)
    cnn_data = np.load(cnn_npz)

    n = min(len(resnet_data["images"]), len(cnn_data["images"]))
    n = min(n, 6)

    fig, axes = plt.subplots(n, 5, figsize=(17, 3 * n))
    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    for i in range(n):
        image = resnet_data["images"][i]
        gt = resnet_data["masks"][i]
        prob_resnet = resnet_data["probs"][i]
        prob_cnn = cnn_data["probs"][i]
        diff = np.abs(prob_resnet - prob_cnn)

        axes[i, 0].imshow(image)
        axes[i, 0].set_title("Image")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(gt, cmap="gray")
        axes[i, 1].set_title("Ground Truth")
        axes[i, 1].axis("off")

        axes[i, 2].imshow(prob_resnet, cmap="viridis", vmin=0.0, vmax=1.0)
        axes[i, 2].set_title("ResNet18 Prob")
        axes[i, 2].axis("off")

        axes[i, 3].imshow(prob_cnn, cmap="viridis", vmin=0.0, vmax=1.0)
        axes[i, 3].set_title("No-Skip CNN Prob")
        axes[i, 3].axis("off")

        axes[i, 4].imshow(diff, cmap="magma")
        axes[i, 4].set_title("Absolute Diff")
        axes[i, 4].axis("off")

    plt.tight_layout()
    qual_path = COMP_PLOT_DIR / "qualitative_side_by_side.png"
    plt.savefig(qual_path, dpi=140, bbox_inches="tight")
    plt.show()
    print(f"Saved qualitative comparison: {qual_path}")

## Decision Guide

- Prefer **ResNet18 (with skip)** if best Dice/IoU and convergence speed are higher.
- Prefer **CNN18 (no skip)** only if it matches quality while offering stability or simplicity advantages in your runs.
- Use the qualitative panel to inspect whether one model over-smooths boundaries or misses thin structures.